# TPCx-AI UC9: Image Classification with Distributed PyTorch

Multi-class image classification using ResNet-50 on the TPCx-AI benchmark (unofficial).

**Pipeline:** Snowflake Stage → Ray Data → PyTorch DDP Training

## Prerequisites

- A Snowflake ML Runtime notebook session with GPU compute enabled.
- GPU instances: Snowflake GPU_NV_M or similar (4 GPUs per node recommended). The notebook will use the `UC9_GPU_POOL` compute pool created by [`setup_snowflake.sql`](./setup_snowflake.sql).
- **Dataset:** TPCx-AI UC9 image classification dataset loaded into Snowflake internal stage.

  **Setup the dataset:**
  1. Generate images locally using TPCx-AI PDGF: [TPCx-AI Specification](https://www.tpc.org/tpc_documents_current_versions/current_specifications5.asp)
  2. Upload images to S3
  3. Review and run [`setup_snowflake.sql`](./setup_snowflake.sql) to load into internal stage
  4. The config cell below is pre-configured to use the stage created by setup script

## Running

1. Open this notebook in a Snowflake ML Runtime session with GPU compute.
2. In the config cell, adjust `NUM_NODES` and training hyperparameters. `DATABASE`/`SCHEMA`/`STAGE_PATH` match the defaults in [`setup_snowflake.sql`](./setup_snowflake.sql) — update if you renamed them there.
3. Run all cells. The Ray cluster scales automatically, then training begins. Loss and timing are printed inline.

## Pipeline Components

- Snowflake Stage — Snowflake-managed internal stage; images served to Ray workers
- `SFStageImageDataSource` — Ray Data source that reads images from stage (no local copy)
- Ray Data — lazy data pipeline for distributed preprocessing and sharding
- `ShardedDataConnector` — automatically partitions Ray Dataset across distributed GPU workers
- PyTorch DataLoader — standard PyTorch data loading (Option A, works for any dataset size)
- Ray Dataset Iterator — streaming data iterator with in-memory caching (Option B, for datasets that fit in memory)
- `PyTorchDistributor` — Snowflake wrapper for launching PyTorch DDP training on Ray cluster
- PyTorch DDP — synchronizes gradients across all GPUs via NCCL

In [ ]:
import datetime

import ray
from ray.data import DataContext

from snowflake.snowpark.context import get_active_session
from snowflake.ml.runtime_cluster import scale_cluster
from snowflake.ml.ray.datasource import SFStageImageDataSource
from snowflake.ml.data.sharded_data_connector import ShardedDataConnector
from snowflake.ml.modeling.distributors.pytorch import (
    PyTorchDistributor,
    PyTorchScalingConfig,
    WorkerResourceConfig,
)

context = DataContext.get_current()
context.execution_options.verbose_progress = False
context.enable_operator_progress_bars = False

In [ ]:
# ---- Cluster ----
NUM_NODES = 1                    # Number of compute nodes
NUM_WORKERS_PER_NODE = 4         # GPUs per node (matches instance GPU count)

# ---- Data ----
DATABASE = "TPCXAI_UC9"
SCHEMA = "PUBLIC"
STAGE_PATH = "UC9_INTERNAL_STAGE/CUSTOMER_IMAGES/"  # Internal stage from setup_snowflake.sql
IMAGE_SIZE = (224, 224)

# ---- Training ----
EPOCHS = 10
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
SEED = 42

# ---- Output ----
MODEL_OUTPUT_PATH = "model.pth"

In [ ]:
scale_cluster(NUM_NODES)
print(f"Ray cluster scaled to {NUM_NODES} node(s)")
print(f"Total GPUs: {NUM_NODES * NUM_WORKERS_PER_NODE}")

## Load Data from Stage

Builds the Ray Data pipeline for distributed image loading.

1. **Discover classes** — Runs LIST @stage to find all image files, extracts class names from directory structure
2. **Define datasource** — SFStageImageDataSource configured to read images from stage
3. **Define label mapping** — Maps directory names to integer labels
4. **Create sharded connector** — ShardedDataConnector configured to partition data across GPU workers

Steps 2-4 are lazy — no data is loaded until training begins.

In [ ]:
session = get_active_session()

# Step 1: Discover class labels from stage directory structure
rows = session.sql(f"LIST @{DATABASE}.{SCHEMA}.{STAGE_PATH}").collect()
class_names = sorted({row["name"].split("/")[-2] for row in rows if row["name"].endswith(".png")})
class_to_idx = {name: i for i, name in enumerate(class_names)}
num_classes = len(class_to_idx)
print(f"Discovered {num_classes} classes, {len(rows):,} files")

# Step 2: Build Ray datasource from stage images
datasource = SFStageImageDataSource(
    stage_location=STAGE_PATH,
    database=DATABASE,
    schema=SCHEMA,
    image_size=list(IMAGE_SIZE),
    image_mode="RGB",
    include_paths=True,
    file_pattern="png",
)

# Step 3: Read images and attach integer labels from directory names
def add_labels(batch):
    batch["label"] = [class_to_idx[p.split("/")[-2]] for p in batch["path"]]
    return batch

ray_ds = ray.data.read_datasource(datasource)
ray_ds = ray_ds.map_batches(add_labels, batch_format="numpy")
ray_ds = ray_ds.drop_columns(["path"])

# Step 4: Create sharded data connector for distributed training
dc = ShardedDataConnector.from_ray_dataset(ray_ds, equal=True)
print("Dataset ready for distributed training.")

## Training Function

Runs on each GPU worker. Supports two data loading approaches:

**Option A: PyTorch DataLoader** (default, active)
- Works for any dataset size
- Standard PyTorch data loading pattern
- Data is streamed from stage on each epoch

**Option B: Ray Dataset Iterator** (commented out)
- For datasets that fit in Ray's object store (~20GB)
- Caches data in memory after first read
- Avoids repeated stage reads across epochs
- Use `shard.to_ray_dataset()` + `iter_torch_batches()`

In [ ]:
def train_func():
    import datetime

    import torch
    import torch.nn as nn
    import torch.optim as optim
    import torch.distributed as dist
    from torch.nn.parallel import DistributedDataParallel as DDP
    from torch.utils.data import DataLoader
    import torchvision.models as models
    import torch.multiprocessing
    from PIL import ImageFile
    from snowflake.ml.modeling.distributors.pytorch import get_context

    torch.multiprocessing.set_sharing_strategy("file_system")
    ImageFile.LOAD_TRUNCATED_IMAGES = True

    ctx = get_context()
    world_size = ctx.get_world_size()
    rank = ctx.get_world_rank()
    local_rank = ctx.get_local_rank()

    def log(msg):
        if rank == 0:
            print(f"[rank {rank}] {msg}", flush=True)

    log(f"world_size={world_size} local_rank={local_rank}")

    # ── Data Loading ─────────────────────────────────────────────────────────
    shard = ctx.get_dataset_map()["input_data"].get_shard()

    # Option A: PyTorch DataLoader (works for any dataset size)
    torch_ds = shard.to_torch_dataset()
    train_loader = DataLoader(
        torch_ds,
        batch_size=BATCH_SIZE,
        drop_last=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    # Option B: Ray Dataset iterator (for ~20GB datasets that fit in memory)
    # Uncomment to use:
    # ray_ds = shard.to_ray_dataset()

    # ── Device setup ─────────────────────────────────────────────────────────
    device = torch.device(f"cuda:{local_rank}" if torch.cuda.is_available() else "cpu")
    if torch.cuda.is_available():
        torch.cuda.set_device(local_rank)

    if world_size > 1 and not dist.is_initialized():
        dist.init_process_group(backend="nccl")

    # ── Model ────────────────────────────────────────────────────────────────
    model = models.resnet50(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    model = model.to(device)
    if world_size > 1:
        model = DDP(model, device_ids=[local_rank], output_device=local_rank)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    # ── Transforms ───────────────────────────────────────────────────────────
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def process_batch(batch):
        images = batch["image"].to(dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
        flip_mask = torch.rand(images.size(0)) > 0.5
        images[flip_mask] = torch.flip(images[flip_mask], dims=[3])
        images = (images - mean) / std
        labels = batch["label"].to(dtype=torch.long).squeeze(-1)
        return images, labels

    # ── Training loop ────────────────────────────────────────────────────────
    log(f"Starting training: {EPOCHS} epochs, batch_size={BATCH_SIZE}")

    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        total = 0

        # Option A: iterate via DataLoader
        for batch in train_loader:
            inputs, labels = process_batch(batch)
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            total += inputs.size(0)

        # Option B: iterate via Ray Dataset iterator (uncomment to use)
        # for batch in ray_ds.iter_torch_batches(batch_size=BATCH_SIZE, prefetch_batches=4):
        #     inputs, labels = process_batch(batch)
        #     inputs = inputs.to(device, non_blocking=True)
        #     labels = labels.to(device, non_blocking=True)
        #
        #     optimizer.zero_grad(set_to_none=True)
        #     outputs = model(inputs)
        #     loss = criterion(outputs, labels)
        #     loss.backward()
        #     optimizer.step()
        #
        #     running_loss += loss.item() * inputs.size(0)
        #     total += inputs.size(0)

        if total > 0:
            log(f"Epoch {epoch}  Loss: {running_loss / total:.4f}")

    log(f"Training complete: {datetime.datetime.now()}")

    if rank == 0:
        state_dict = model.module.state_dict() if hasattr(model, "module") else model.state_dict()
        torch.save(state_dict, MODEL_OUTPUT_PATH)
        log(f"Model saved to {MODEL_OUTPUT_PATH}")

    if dist.is_initialized():
        dist.destroy_process_group()

## Launch Distributed Training

`PyTorchDistributor` spawns one worker process per GPU across all Ray cluster nodes. Each worker trains a replica of the model, and DDP synchronizes gradients after each backward pass via NCCL.

In [ ]:
trainer = PyTorchDistributor(
    train_func=train_func,
    scaling_config=PyTorchScalingConfig(
        num_nodes=NUM_NODES,
        num_workers_per_node=NUM_WORKERS_PER_NODE,
        resource_requirements_per_worker=WorkerResourceConfig(num_cpus=1, num_gpus=1),
    ),
)

start_ts = datetime.datetime.now()
print(f"Training started: {start_ts}")
print(f"Config: {NUM_NODES} node(s), {NUM_WORKERS_PER_NODE} GPUs/node, {EPOCHS} epochs, batch_size={BATCH_SIZE}")

trainer.run(dataset_map={"input_data": dc})

end_ts = datetime.datetime.now()
elapsed = (end_ts - start_ts).total_seconds()
print(f"\nTraining complete: {end_ts}")
print(f"Total time: {elapsed:.1f}s ({elapsed/60:.1f} min)")